# P6(검색 확인). 대표 질문 Q01·Q07·Q09 — 테스트 DB 검색

## 이 노트북이 하는 일
- 블록 16개를 **테스트 라벨 `PLTest`와 `test_` 접두사**로 공동 DB에 넣습니다.
- 대표 질문 3개에 대해 **키워드 점수**로 블록을 찾고, 기대한 블록이 상위에 오는지 봅니다.
- 끝나면 테스트 데이터를 **삭제**하고 0개인지 확인합니다.

## 주의
- 이 검색은 **단순 키워드 점수**입니다. 실제 검색 방식(전문 검색, 벡터 검색)은 나중에 비교합니다.
- 공동 DB에 **인덱스는 만들지 않습니다.** 테스트 노드 몇 개를 파이썬에서 점수 계산합니다.
- 결과는 "후보" 확인입니다. 최종 근거는 사람이 원문으로 확인합니다.

In [1]:
import json
from rag.prevention_legal.neo4j_conn import use_windows_trust, connect

use_windows_trust()
driver = connect()
TEST_LABEL, PREFIX = "PLTest", "test_"

def count_all():
    with driver.session() as s:
        return s.run("MATCH (n) RETURN count(n) AS c").single()["c"]

print("시작 전 전체 노드:", count_all())

시작 전 전체 노드: 0


## 1. 블록 적재 (테스트 라벨 + 접두사)

In [2]:
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
blocks = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]

CYPHER_LOAD = f"""
MERGE (c:{TEST_LABEL} {{id: $id}})
SET c.document_id = $document_id, c.pdf_page = $pdf_page,
    c.section = $section, c.text = $text, c.is_test = true
"""
with driver.session() as s:
    for b in blocks:
        s.run(CYPHER_LOAD, id=PREFIX + b["chunk_id"], document_id=b["document_id"],
              pdf_page=b["pdf_page"], section=b.get("section") or "", text=b["text"])

with driver.session() as s:
    n = s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p RETURN count(c) AS c", p=PREFIX).single()["c"]
print("테스트 블록 적재:", n, "개 (블록 파일:", len(blocks), ")")

테스트 블록 적재: 16 개 (블록 파일: 16 )


## 2. 대표 질문별 검색 (키워드 점수)
- 질문마다 **검색어 목록**을 정합니다. 블록 글과 소제목에 검색어가 몇 번 나오는지 셉니다.
- **기대 블록**: 그 질문의 정답 근거로 기대하는 블록입니다. 상위 5개 안에 있어야 합니다.

In [3]:
# 대표 질문: (ID, 질문, 검색어, 기대 블록)
QUESTIONS = [
    ("Q01", "교내 홍역 의심환자가 생기면 담당교사는 진료와 보건소 신고에 관해 무엇을 해야 하나요?",
     ["홍역", "진료", "보건소", "신고"],
     "kdca_measles_2026:p0093:b01"),
    ("Q07", "홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇을 신고하거나 안내하나요?",
     ["의료기관", "보건소", "신고", "홍역"],
     "kdca_measles_2026:p0093:b01"),
    ("Q09", "감염병예방법상 학교의 관리인이나 대표자도 신고해야 하는 경우가 있나요?",
     ["관리인", "대표자", "학교", "신고"],
     "moleg_infectious_disease_act_21521:p0019:b02"),
]

with driver.session() as s:
    all_rows = s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p RETURN c.id AS id, c.text AS text, c.section AS section",
                     p=PREFIX).data()

def score(row, words):
    # 블록 글 + 소제목에서 검색어가 나온 횟수의 합
    hay = (row["text"] or "") + " " + (row["section"] or "")
    return sum(hay.count(w) for w in words)

for qid, question, words, expected in QUESTIONS:
    ranked = sorted(all_rows, key=lambda r: score(r, words), reverse=True)
    top5 = [r["id"][len(PREFIX):] for r in ranked[:5]]
    hit = expected in top5
    print(f"[{qid}] 기대 블록 상위 5개 안: {'있음 ✅' if hit else '없음 ❌'}  ({expected})")
    for r in ranked[:3]:
        print(f"    점수 {score(r, words):2d} | {r['id'][len(PREFIX):]}")

[Q01] 기대 블록 상위 5개 안: 있음 ✅  (kdca_measles_2026:p0093:b01)
    점수  9 | kdca_diagnosis_reporting_2026:p0089:b04
    점수  8 | kdca_diagnosis_reporting_2026:p0104:b04
    점수  5 | kdca_diagnosis_reporting_2026:p0089:b02
[Q07] 기대 블록 상위 5개 안: 있음 ✅  (kdca_measles_2026:p0093:b01)
    점수  9 | kdca_diagnosis_reporting_2026:p0089:b04
    점수  8 | kdca_diagnosis_reporting_2026:p0104:b04
    점수  5 | kdca_diagnosis_reporting_2026:p0089:b02
[Q09] 기대 블록 상위 5개 안: 있음 ✅  (moleg_infectious_disease_act_21521:p0019:b02)
    점수  7 | kdca_diagnosis_reporting_2026:p0089:b04
    점수  7 | kdca_diagnosis_reporting_2026:p0104:b04
    점수  3 | kdca_measles_2026:p0093:b01


## 3. 정리: 테스트 데이터 삭제
접두사와 테스트 라벨이 **둘 다** 붙은 노드만 지우고, 0개인지 확인합니다.

In [4]:
with driver.session() as s:
    s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p DETACH DELETE c", p=PREFIX)
    left = s.run(f"MATCH (c:{TEST_LABEL}) RETURN count(c) AS c").single()["c"]
print("삭제 후 테스트 노드:", left, "| 전체 노드:", count_all())
driver.close()

삭제 후 테스트 노드: 0 | 전체 노드: 0
